# Day 047 Project: Stats Report

## What You're Building

A full statistical analysis of a multi-column dataset using `StatsReport`, `correlation_with_pvalue`, and `compare_groups`. The report is saved as a matplotlib figure with distribution plots and annotated group comparison.

## Project Requirements

1. Generate a dataset with at least 100 rows using `make_sample_data()`
2. Run `StatsReport().load(df).report()` and store as `report`
3. Print the skewness and is_normal flag for every column
4. Run `compare_groups(df['score_a'], df['score_b'])` and store as `comparison`
5. Run `correlation_with_pvalue` between at least one pair of columns
6. Save a figure with at least 2 subplots (e.g. histograms, box plot)
7. Run `_run_project_checks()` to verify

You run it, it prints a stats summary and saves a chart. That is the deliverable.

## Provided: All Implementations

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def make_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible multi-column dataset for statistics exercises."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'normal_col': rng.standard_normal(n).round(3),
        'skewed_col': rng.exponential(2, n).round(3),
        'score_a':    (50 + rng.standard_normal(n) * 10).round(1),
        'score_b':    (70 + rng.standard_normal(n) * 10).round(1),
    })


def describe_distribution(series: pd.Series) -> dict:
    s   = series.dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    return {
        'count':    int(len(s)),
        'mean':     round(float(s.mean()), 4),
        'median':   round(float(s.median()), 4),
        'std':      round(float(s.std(ddof=1)), 4),
        'sem':      round(float(s.sem()), 4),
        'min':      round(float(s.min()), 4),
        'max':      round(float(s.max()), 4),
        'q25':      round(q25, 4),
        'q75':      round(q75, 4),
        'iqr':      round(q75 - q25, 4),
        'skewness': round(float(s.skew()), 4),
        'kurtosis': round(float(s.kurt()), 4),
    }


def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    s      = series.dropna()
    stat, p = stats.shapiro(s)
    return {
        'n':          len(s),
        'statistic':  round(float(stat), 4),
        'p_value':    round(float(p), 6),
        'is_normal':  bool(p > alpha),
        'alpha':      alpha,
    }


def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    if method == 'pearson':
        r, p = stats.pearsonr(x_c, y_c)
    elif method == 'spearman':
        r, p = stats.spearmanr(x_c, y_c)
    else:
        raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    return {
        'method':         method,
        'n':              len(x_c),
        'r':              round(float(r), 4),
        'p_value':        round(float(p), 6),
        'is_significant': bool(p < 0.05),
    }


def compare_groups(a: pd.Series, b: pd.Series,
                   alpha: float = 0.05) -> dict:
    a_c, b_c  = a.dropna(), b.dropna()
    t, p       = stats.ttest_ind(a_c, b_c)
    n_a, n_b   = len(a_c), len(b_c)
    std_a      = float(a_c.std(ddof=1))
    std_b      = float(b_c.std(ddof=1))
    pooled_var = ((n_a - 1) * std_a**2 + (n_b - 1) * std_b**2) / (n_a + n_b - 2)
    pooled     = np.sqrt(pooled_var) if pooled_var > 0 else 0.0
    d          = (float(a_c.mean()) - float(b_c.mean())) / pooled if pooled > 0 else 0.0
    sig        = bool(p < alpha)
    return {
        'n_a':            n_a,
        'n_b':            n_b,
        'mean_a':         round(float(a_c.mean()), 4),
        'mean_b':         round(float(b_c.mean()), 4),
        'statistic':      round(float(t), 4),
        'p_value':        round(float(p), 6),
        'is_significant': sig,
        'cohens_d':       round(d, 4),
        'conclusion':     'different' if sig else 'not_different',
    }


class StatsReport:
    def __init__(self):
        self._df      = None
        self._columns = None

    def load(self, df: pd.DataFrame,
             columns: list | None = None) -> 'StatsReport':
        self._df      = df.copy()
        num_cols      = df.select_dtypes(include='number').columns.tolist()
        self._columns = columns if columns is not None else num_cols
        return self

    def report(self) -> dict:
        result = {}
        for col in self._columns:
            if col not in self._df.columns:
                continue
            s = self._df[col].dropna()
            if len(s) < 3:
                continue
            result[col] = {
                'distribution': describe_distribution(s),
                'normality':    test_normality(s),
            }
        return result

## Your Pipeline

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = make_sample_data(n=100, seed=42)

# TODO: create and run StatsReport
# report = StatsReport().load(df).report()
# for col, entry in report.items():
#     d = entry['distribution']
#     n = entry['normality']
#     print(f"{col}: skew={d['skewness']:.2f} is_normal={n['is_normal']}")

# TODO: compare the two score groups
# comparison = compare_groups(df['score_a'], df['score_b'])
# print(comparison)

# TODO: compute at least one correlation
# corr_result = correlation_with_pvalue(df['score_a'], df['score_b'])
# print(corr_result)

# TODO: save a 2-subplot figure to 'stats_report.png'
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ax1.hist(df['normal_col'], bins=20, edgecolor='white')
# ax1.set_title('Normal Column')
# ax2.hist(df['skewed_col'], bins=20, edgecolor='white', color='orange')
# ax2.set_title('Skewed Column')
# fig.savefig('stats_report.png', bbox_inches='tight', dpi=100)
# plt.close('all')
# print('Chart saved: stats_report.png')

## Checks

In [ ]:
import os

def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: report defined as dict
    try:
        assert 'report' in globals(), 'report not defined — run StatsReport'
        assert isinstance(report, dict) and len(report) > 0
        passed += 1; print('\u2705 Check 1: report is a non-empty dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: report has distribution and normality for each col
    try:
        for col, entry in report.items():
            assert 'distribution' in entry
            assert 'normality' in entry
        passed += 1; print('\u2705 Check 2: each column has distribution + normality')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: comparison defined as dict
    try:
        assert 'comparison' in globals(), 'comparison not defined — run compare_groups'
        assert isinstance(comparison, dict)
        assert 'conclusion' in comparison
        passed += 1; print(f'\u2705 Check 3: comparison conclusion={comparison["conclusion"]!r}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: corr_result defined
    try:
        assert 'corr_result' in globals(), 'corr_result not defined — run correlation_with_pvalue'
        assert isinstance(corr_result, dict)
        assert 'r' in corr_result
        passed += 1; print(f'\u2705 Check 4: corr_result r={corr_result["r"]}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: chart file saved
    try:
        assert os.path.exists('stats_report.png'), \
            'stats_report.png not found — save with fig.savefig()'
        assert os.path.getsize('stats_report.png') > 1000, \
            'stats_report.png looks empty'
        passed += 1; print('\u2705 Check 5: stats_report.png saved')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Test normality before using correlation: if either variable is non-normal, use Spearman instead of Pearson
- Create a full correlation matrix heatmap with `ax.imshow()` using `df.corr()`
- Add a third group column and use `compare_groups` on all pairs
- Use `ollama.chat` (Day 40 pattern) to narrate the `StatsReport` output in plain English
- Export the report as a JSON file with `json.dumps(report, default=str, indent=2)`